# Update meta data and objects 

In [ ]:
import scvi
import scanpy as sc

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

import os
import sys

## Root directory

In [ ]:
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

## Custom modules

In [ ]:
sys.path.append('bin/')
from adata_qc import *

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.4.1-FD20250213HSCT/lib/R'

In [ ]:
import rpy2
%load_ext rpy2.ipython

## Figures 

In [ ]:
sc.set_figure_params(figsize=(5, 5), dpi_save=1200, fontsize=12, frameon=False, facecolor='white')
mpl.rcParams['figure.facecolor'] = 'white'

# Import 

In [ ]:
adata = sc.read_h5ad('data/object/pp.h5ad')

In [ ]:
sc.pl.umap(adata, color=['celltype_low', 'celltype_high', 'celltype_domain', 'patient_id', 'hto_demux_class', 'time_point', 'msCC_label', 'genotype_class', 'clone_id_size'], size=20, ncols=5, wspace=0.1, legend_loc='on data')

# SCVI on T cells/NK

In [ ]:
adata_0 = adata[adata.obs.celltype_high.isin(["$T\ cells$", "$NK\ cells$"])]
adata_0 = adata_0[:, (adata_0.X>=3).sum(axis=0)>=3]

In [ ]:
adata_0.obs['labels_key'] = (adata_0.obs['celltype_low'].astype(str).str.cat(adata_0.obs['hto_demux_class'].astype(str), sep="_"))

In [ ]:
adata_0 = adata_0.copy()

In [ ]:
cache_scvi = True

In [ ]:
if not cache_scvi: 
    
    scvi.model.SCVI.setup_anndata(
    
        adata_0,
        batch_key='sample_name', 
        continuous_covariate_keys=['S_ucell', 'G2M_ucell'], 
        labels_key='labels_key'
        
    )
    
    model_0 = scvi.model.SCVI(
    
        adata_0, 
        n_latent=30, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'

    )
    
    max_epochs = int(np.min([round((20000 / adata_0.n_obs) * 400), 400]))
    
    model_0.train(max_epochs=max_epochs, check_val_every_n_epoch=1)
    
    model_0.save('data/object/scvi/subset/model_0/', overwrite=True)
    
else: 
    
    model_0 = scvi.model.SCVI.load('data/object/scvi/subset/model_0/', adata=adata_0)

In [ ]:
cache_dim = True

In [ ]:
if not cache_dim: 
    
    adata_0.obsm['latent'] = model_0.get_latent_representation()
    
    sc.pp.neighbors(adata_0, n_neighbors=15, use_rep='latent')
    sc.tl.umap(adata_0, min_dist=1)
    
    adata_0.write_h5ad('data/object/scvi/subset/adata_0.h5ad')
    
else: 
    
    adata_0 = sc.read_h5ad('data/object/scvi/subset/adata_0.h5ad')

In [ ]:
ir = pd.read_csv('result/ir/ir.csv', index_col=0)

In [ ]:
adata_0.obs = adata_0.obs.drop(columns=adata_0.obs.columns.intersection(ir.columns))
adata_0.obs = adata_0.obs.merge(ir, left_index=True, right_index=True, how="left")

In [ ]:
sc.pp.normalize_total(adata_0)
sc.pp.log1p(adata_0)

In [ ]:
sc.pl.umap(adata_0, color=['celltype_low', 'celltype_high', 'CD4', 'CD8A', 'CD8B', 'celltype_domain', 'patient_id', 'hto_demux_class', 'time_point', 'msCC_label', 'genotype_class', 'clone_id_size'], size=50, ncols=5, wspace=0.1, legend_loc='on data', use_raw=False)